# Ngô Minh Khôi - 6451071037
## Bai2 - Câu 6

In [1]:
# Ngô Minh Khôi - 6451071037
# Bài 2 - Câu 6: Tiền xử lý dữ liệu và tạo thuộc tính mới
import pandas as pd
import numpy as np
import re

print("Ngô Minh Khôi - 6451071037")
df = pd.read_csv(r"dataset/titanic.csv")

# 1. Trích xuất đặc trưng deck từ cabin và loại bỏ cabin
deck = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "U": 8}
df['cabin'] = df['cabin'].fillna("U0")
df['deck'] = df['cabin'].map(lambda x: re.compile("([a-zA-Z]+)").search(x).group())
df['deck'] = df['deck'].map(deck)
df['deck'] = df['deck'].fillna(0).astype(int)
df.drop(columns=['cabin'], inplace=True)

# 2. Xóa các thuộc tính không cần thiết
df.drop(columns=['ticket', 'boat', 'body', 'home.dest'], inplace=True, errors='ignore')

# 3. Điền giá trị thiếu cho thuộc tính tuổi (age) ngẫu nhiên quanh [mean - std, mean + std]
np.random.seed(42)
mean_age = df['age'].mean()
std_age = df['age'].std()
is_null_age = df['age'].isnull().sum()
rand_age = np.random.randint(int(mean_age - std_age), int(mean_age + std_age), size=is_null_age)
age_slice = df['age'].copy()
age_slice[np.isnan(age_slice)] = rand_age
df['age'] = age_slice.astype(int)

# 4. Điền giá trị thiếu cho Cổng lên tàu (embarked) bằng giá trị phổ biến nhất ('S')
df['embarked'] = df['embarked'].fillna('S')

# 5. Điền giá trị thiếu cho giá vé (fare) và ép kiểu int
df['fare'] = df['fare'].fillna(df['fare'].median()).astype(int)

# 6. Trích xuất danh hiệu (title) từ tên (name) và mã hóa thành số, sau đó drop name
titles = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
df['title'] = df['name'].str.extract(' ([A-Za-z]+)\.', expand=False)
df['title'] = df['title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['title'] = df['title'].replace('Mlle', 'Miss')
df['title'] = df['title'].replace('Ms', 'Miss')
df['title'] = df['title'].replace('Mme', 'Mrs')
df['title'] = df['title'].map(titles).fillna(0).astype(int)
df.drop(columns=['name'], inplace=True)

# 7. Chuyển đổi giới tính (sex) và cổng lên tàu (embarked) thành số
df['sex'] = df['sex'].map({'male': 0, 'female': 1}).astype(int)
df['embarked'] = df['embarked'].map({'S': 0, 'C': 1, 'Q': 2}).astype(int)

# 8. Chia nhóm tuổi (age) thành 8 nhóm cân bằng
df.loc[df['age'] <= 11, 'age'] = 0
df.loc[(df['age'] > 11) & (df['age'] <= 18), 'age'] = 1
df.loc[(df['age'] > 18) & (df['age'] <= 22), 'age'] = 2
df.loc[(df['age'] > 22) & (df['age'] <= 27), 'age'] = 3
df.loc[(df['age'] > 27) & (df['age'] <= 33), 'age'] = 4
df.loc[(df['age'] > 33) & (df['age'] <= 40), 'age'] = 5
df.loc[(df['age'] > 40) & (df['age'] <= 66), 'age'] = 6
df.loc[df['age'] > 66, 'age'] = 7

# 9. Chia nhóm giá vé (fare) thành 6 nhóm
df.loc[df['fare'] <= 7.91, 'fare'] = 0
df.loc[(df['fare'] > 7.91) & (df['fare'] <= 14.454), 'fare'] = 1
df.loc[(df['fare'] > 14.454) & (df['fare'] <= 31), 'fare'] = 2
df.loc[(df['fare'] > 31) & (df['fare'] <= 99), 'fare'] = 3
df.loc[(df['fare'] > 99) & (df['fare'] <= 250), 'fare'] = 4
df.loc[df['fare'] > 250, 'fare'] = 5

# 10. Tạo các thuộc tính mới
df['relatives'] = df['sibsp'] + df['parch']
df['not_alone'] = (df['relatives'] == 0).astype(int)
df['age_class'] = df['age'] * df['pclass']
df['fare_per_person'] = (df['fare'] / (df['relatives'] + 1)).astype(int)

print("--- 10 dòng đầu dữ liệu sau tiền xử lý hoàn chỉnh ---")
print(df.head(10))

print("\n--- Thông tin các cột sau khi tiền xử lý ---")
print(df.info())
print("\nTổng số giá trị null còn lại:", df.isnull().sum().sum())


Ngô Minh Khôi - 6451071037
--- 10 dòng đầu dữ liệu sau tiền xử lý hoàn chỉnh ---
   pclass  survived  sex  ...  not_alone  age_class  fare_per_person
0       1         1    1  ...          1          4                4
1       1         1    0  ...          0          0                1
2       1         0    1  ...          0          0                1
3       1         0    0  ...          0          4                1
4       1         0    1  ...          0          3                1
5       1         1    0  ...          1          6                2
6       1         1    1  ...          0          6                1
7       1         0    0  ...          1          5                0
8       1         1    1  ...          0          6                1
9       1         0    0  ...          1          7                3

[10 rows x 14 columns]

--- Thông tin các cột sau khi tiền xử lý ---
<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 colu